# Cross Encoder testing
Test the performance of cross encoder in pairing soilvoc keywords with record metadata (title, abstract, pdf, ...)

In [20]:
import json, time
import torch
from sentence_transformers import CrossEncoder

DOC = """Relative Contribution Of Trees And Crops To Soil Carbon Content In A Parkland System In Burkina Faso Using Variations In Natural C-13 Abundance","The Origin Of Organic Matter Was Studied In The Soils Of A Parkland Of Karite (Vitallaria Paradoxa C.F. Gaertn) And Nere (Parkia Biglobosa (Jacq.) Benth.), Which Is Extensively Cultivated Without The Use Of Fertilisers. In Such Systems, Fertility (Physical, Chemical And Biological) Gradients Around Trees Have Been Attributed By Some Authors To A Priori Differences In Fertility, Allowing For Better Tree Establishment On Richer Sites. In Reverse, Other Workers Believed That These Gradients Are Due To The Contribution Of Trees To The Formation Of Soil Organic Matter Through Litter And Decay Of Roots. Measurements Of The Variations In The C-13 Isotopic Composition Allowed For A Distinction Between Tree (C-3) Derived C And Crop And Grass (C-4) Derived C In The Total Soil Organic C Content. The Organic Carbon Contents Of The Soils Were Recorded Under The Two Species At Two Soil Depths And At Five Distances Going From Tree Trunk To The Open Area And Their C Isotopic Signatures Were Analysed. The Results Showed That Soil Carbon Contents Under Karite (6.43 +/- 0.45 G Kg(-1)) And Nere (5.65 +/- 0.27 G Kg(-1)) Were Significantly Higher (P < 0.01) Than In The Open Area (4.09 +/- 0.26 G Kg(-1)). The Delta C-13 Of Soil C Was Significantly Higher (P < 0.001) In The Open Area (-17.5 +/- 0.3 Parts Per Thousand) Compared With The Values Obtained On Average With Depth And Distance From Tree Under Karite (-20.2 +/- 0.4 Parts Per Thousand) And Nere (-20.1 +/- 0.4 Parts Per Thousand). The C-4-Derived Soil C Was Approximately Constant, And The Differences In Total Soil C Were Fully Explained By The C-3 (Tree) Contributions To Soil Carbon Of 4.01 +/- 0.71, 3.02 +/- 0.53, 1.53 +/- 0.10 G Kg(-1), Respectively Under Karite, Nere And In The Open Area. These Results Show That Trees In Parklands Have A Directly Positive Contribution To Soil Carbon Content, Justifying The Need To Encourage The Maintenance Of Trees In These Systems In Semi-Arid Environments Where The Carbon Content Of Soil Appears To Be The First Limiting Factor For Crop Growth.
"""

with open("../concepts_multilingual.json", encoding="utf-8") as f:
    concepts = json.load(f)

# One (label, concept) entry per English label; several labels share a concept.
labels = [(lab, c["identifier"].split("#")[-1])
          for c in concepts for lab in c["labels"].get("en", [])]
print(f"{len(labels)} English labels from {len(concepts)} concepts")

model = CrossEncoder("cross-encoder/mmarco-mMiniLMv2-L12-H384-v1",
                     activation_fn=torch.nn.Sigmoid(), max_length=256)
scores = model.predict([(lab, DOC) for lab, _ in labels], show_progress_bar=True)
for score, (lab, cid) in sorted(zip(scores, labels), reverse=True)[:10]:
    print(f"  {score:.4f}  {lab:<32} {cid}")


1064 English labels from 799 concepts


Batches: 100%|██████████| 34/34 [01:38<00:00,  2.89s/it]

  0.8791  soil organic matter content      SoilOrganicMatterContents
  0.8339  soil organic matter contents     SoilOrganicMatterContents
  0.7613  soil organic matter              SoilOrganicMatter
  0.7025  soil organic carbon              SoilOrganicCarbon
  0.4383  soil organic components          SoilOrganicComponents
  0.4254  soil organic component           SoilOrganicComponents
  0.2185  critical soil organic matter content CriticalSoilOrganicMatterContents
  0.1831  soil inorganic carbon            SoilInorganicCarbon
  0.1815  soil organic matter class        SoilOrganicMatterClass
  0.1516  critical soil organic matter contents CriticalSoilOrganicMatterContents


In [ ]:
# Now try with the german content.
DOC_DE = """
Die Gesamt-Phosphoreinträge in die Gewässer wurden mit dem Stoffflussmodell MODIFFUS über alle diffusen Eintragsquellen (Ackerland, Dauergrünland, Wald, Gletscher, Siedlungsgrünflächen etc.) und alle diffusen Eintragspfade (Bodenerosion, Auswaschung, Abschwemmung, Drainage, atmosphärische Deposition etc.) berechnet. Die Karte zeigt die aufsummierten Verluste pro Landnutzungskategorie im Hektarraster, basierend auf der Arealstatistik 2013/18. Es wurden mittlere klimatische Bedingungen zugrunde gelegt, das Bezugsjahr ist 2020.
"""
scores = model.predict([(lab, DOC_DE) for lab, _ in labels], show_progress_bar=True)

for score, (lab, cid) in sorted(zip(scores, labels), reverse=True)[:10]:
    print(f"  {score:.4f}  {lab:<32} {cid}")


Batches: 100%|██████████| 34/34 [00:58<00:00,  1.72s/it]

  0.8466  phosphorus total elements        PhosphorusTotalElements
  0.6913  soil phosphorus loss             SoilPLoss
  0.5186  soil water loss                  SoilWaterLoss
  0.4520  soil P loss                      SoilPLoss
  0.4472  soil water deficit               SoilMoistureDeficit
  0.4143  soil particle movement           SoilParticleMovement
  0.4046  land use class                   LandUseClass
  0.3834  soil organic carbon loss         SOCLoss
  0.3286  soil water contents              SoilWaterContents
  0.3022  soil oxygen contents             SoilOxygenContents


CE doing better than KeyBERT in pairing content with keywords in different languages (de - en). The result looks ok, Phosphorus (P) is not there but phosphorus total elements is there.

In [ ]:
# "Bi-encoder retrieval + cross-encoder rerank for a better efficiency.
# set up
import re
import numpy as np
from sentence_transformers import SentenceTransformer

CONCEPTS_PATH = "../concepts_multilingual.json"
BI_MODEL = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
CE_MODEL = "cross-encoder/mmarco-mMiniLMv2-L12-H384-v1"


def load_vocab(path=CONCEPTS_PATH, lang="en"):
    """-> (labels, concept_ids): one entry per label; a concept may have several."""
    with open(path, encoding="utf-8") as f:
        concepts = json.load(f)
    pairs = [(lab, c["identifier"].split("#")[-1])
             for c in concepts for lab in c["labels"].get(lang, [])]
    labels, concept_ids = map(list, zip(*pairs))
    print(f"{len(labels)} {lang} labels from {len(concepts)} concepts")
    return labels, concept_ids


LABELS, CONCEPT_IDS = load_vocab()
bi = SentenceTransformer(BI_MODEL)
ce = CrossEncoder(CE_MODEL, activation_fn=torch.nn.Sigmoid(), max_length=256)
LABEL_EMB = bi.encode(LABELS, batch_size=128, normalize_embeddings=True,
                      show_progress_bar=True)   # 1064 x 384, reused for every doc

def chunk_text(text, lo=200, hi=900):
    """Sentence-split, then merge back into blocks of roughly lo..hi characters."""
    text = re.sub(r"\s+", " ", text).strip()
    chunks, buf = [], ""
    for sent in re.split(r"(?<=[.!?])\s+(?=[A-ZÄÖÜ])", text):
        if buf and len(buf) + len(sent) + 1 > hi:
            chunks.append(buf)
            buf = sent
        else:
            buf = f"{buf} {sent}".strip()
        if len(buf) >= lo:
            chunks.append(buf)
            buf = ""
    if buf:                      # trailing fragment joins the last chunk
        if chunks and len(buf) < lo:
            chunks[-1] += " " + buf
        else:
            chunks.append(buf)
    return chunks


def retrieve(chunks, top_k=20):
    """Stage 1 — bi-encoder shortlist: top_k labels per chunk.
    -> (pairs, cos) where pairs is [(chunk_index, label_index), ...]"""
    cos = bi.encode(chunks, normalize_embeddings=True) @ LABEL_EMB.T
    k = min(top_k, len(LABELS) - 1)
    pairs = [(ci, int(li)) for ci in range(len(chunks))
             for li in np.argpartition(-cos[ci], k)[:k]]
    return pairs, cos


def rerank(chunks, pairs, cos, batch_size=32, progress=False):
    """Stage 2 — cross-encoder scores each (label, chunk) pair, best kept per concept."""
    scores = ce.predict([(LABELS[li], chunks[ci]) for ci, li in pairs],
                        batch_size=batch_size, show_progress_bar=progress)
    best = {}
    for (ci, li), s in zip(pairs, scores):
        cid = CONCEPT_IDS[li]
        if float(s) > best.get(cid, {}).get("ce", -1):
            best[cid] = {"concept": cid, "label": LABELS[li], "ce": float(s),
                         "cos": float(cos[ci, li]), "chunk": ci}
    return sorted(best.values(), key=lambda r: -r["ce"])


def rank_concepts(doc, top_k=20, lo=200, hi=900, verbose=True):
    """Full pipeline for one record -> ranked list of dicts."""
    t0 = time.time()
    chunks = chunk_text(doc, lo, hi)
    pairs, cos = retrieve(chunks, top_k)
    ranked = rerank(chunks, pairs, cos, progress=verbose)
    if verbose:
        n_lab = len({li for _, li in pairs})
        print(f"{len(doc)} chars -> {len(chunks)} chunks | {len(pairs)} CE pairs, "
              f"{n_lab} labels = {n_lab / len(LABELS):.0%} of vocab | "
              f"{time.time() - t0:.1f}s")
    return ranked


def show(ranked, n=15):
    print(f"  {'ce':>7} {'cos':>6} {'chk':>3}  label")
    for r in ranked[:n]:
        print(f"  {r['ce']:7.4f} {r['cos']:6.3f} {r['chunk']:>3}  "
              f"{r['label']:<34} {r['concept']}")




1064 en labels from 799 concepts


Batches: 100%|██████████| 9/9 [00:02<00:00,  4.25it/s]


In [ ]:
# retrival + reranking test for en DOC

show(rank_concepts(DOC), n=10)                     

Batches: 100%|██████████| 5/5 [00:03<00:00,  1.58it/s]

2196 chars -> 8 chunks | 160 CE pairs, 92 labels = 9% of vocab | 3.4s
       ce    cos chk  label
   0.9939  0.749   3  soil organic carbon                SoilOrganicCarbon
   0.9769  0.616   3  soil organic matter content        SoilOrganicMatterContents
   0.9555  0.731   6  soil total carbon                  SoilTotalCarbon
   0.8216  0.693   4  soil carbon density                SoilCarbonDensity
   0.8028  0.627   3  soil organic component             SoilOrganicComponents
   0.7475  0.616   2  soil organic matter                SoilOrganicMatter
   0.6906  0.703   3  soil organic carbon loss           SOCLoss
   0.5926  0.720   3  soil inorganic carbon              SoilInorganicCarbon
   0.5749  0.607   3  critical soil organic matter content CriticalSoilOrganicMatterContents
   0.5098  0.483   1  fertiliser use                     FertiliserUse


To Compare with the approach: cross encoder only. Results are similar. 

In [ ]:
DOC_DE = """
Die Gesamt-Phosphoreinträge in die Gewässer wurden mit dem Stoffflussmodell MODIFFUS über alle diffusen Eintragsquellen (Ackerland, Dauergrünland, Wald, Gletscher, Siedlungsgrünflächen etc.) und alle diffusen Eintragspfade (Bodenerosion, Auswaschung, Abschwemmung, Drainage, atmosphärische Deposition etc.) berechnet. Die Karte zeigt die aufsummierten Verluste pro Landnutzungskategorie im Hektarraster, basierend auf der Arealstatistik 2013/18. Es wurden mittlere klimatische Bedingungen zugrunde gelegt, das Bezugsjahr ist 2020.
"""
show(rank_concepts(DOC_DE), n = 10)  

Batches: 100%|██████████| 2/2 [00:00<00:00,  2.63it/s]

532 chars -> 2 chunks | 40 CE pairs, 40 labels = 4% of vocab | 1.0s
       ce    cos chk  label
   0.5546  0.420   1  land use class                     LandUseClass
   0.3380  0.630   0  soil water contents                SoilWaterContents
   0.2571  0.623   0  soil water flow                    SoilWaterFlow
   0.2232  0.632   0  soil water content                 SoilMoisture
   0.1505  0.616   0  soil water condensation            SoilWaterCondensation
   0.1465  0.419   1  SOC loss                           SOCLoss
   0.1211  0.625   0  water transfer (in soil)           SoilWaterMovement
   0.1085  0.616   0  soil water infiltration            SoilWaterInfiltration
   0.0831  0.628   0  soil water diffusivity             SoilWaterDiffusivity
   0.0636  0.438   1  soil P loss                        SoilPLoss


Summary: The retrieve-rerankng approach performs good on english content, bad on germany content. 

Now we try to use different embedding models and compare the results.

In [22]:
import json, re, time
import numpy as np
import torch
from sentence_transformers import SentenceTransformer, CrossEncoder

CONCEPTS_PATH = "../concepts_multilingual.json"
BI_MODEL = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
CE_MODEL = "cross-encoder/mmarco-mMiniLMv2-L12-H384-v1"

# Models that expect an instruction prefix. label_ = the vocabulary side,
# text_ = the record side. E5 scores badly without these.
PREFIXES = {
    "intfloat/multilingual-e5-small": ("passage: ", "query: "),
    "intfloat/multilingual-e5-base":  ("passage: ", "query: "),
    "BAAI/bge-m3":                    ("", ""),
}


def load_vocab(path=CONCEPTS_PATH, lang="en"):
    """-> (labels, concept_ids): one entry per label; a concept may have several."""
    with open(path, encoding="utf-8") as f:
        concepts = json.load(f)
    pairs = [(lab, c["identifier"].split("#")[-1])
             for c in concepts for lab in c["labels"].get(lang, [])]
    labels, concept_ids = map(list, zip(*pairs))
    print(f"{len(labels)} {lang} labels from {len(concepts)} concepts")
    return labels, concept_ids


LABELS, CONCEPT_IDS = load_vocab()

_bi_cache, _ce_cache, _emb_cache = {}, {}, {}


def get_bi(name=None):
    name = name or BI_MODEL
    if name not in _bi_cache:
        print(f"loading bi-encoder {name} …")
        _bi_cache[name] = SentenceTransformer(name)
    return _bi_cache[name]


def get_ce(name=None, max_length=256):
    name = name or CE_MODEL
    key = (name, max_length)
    if key not in _ce_cache:
        print(f"loading cross-encoder {name} (max_length={max_length}) …")
        _ce_cache[key] = CrossEncoder(name, activation_fn=torch.nn.Sigmoid(),
                                      max_length=max_length)
    return _ce_cache[key]


def get_label_emb(name=None, progress=True):
    """Encode the vocabulary once per (model, label set). Cheap on repeat calls."""
    name = name or BI_MODEL
    key = (name, len(LABELS), hash(tuple(LABELS)))
    if key not in _emb_cache:
        lab_pre, _ = PREFIXES.get(name, ("", ""))
        print(f"encoding {len(LABELS)} labels with {name} …")
        _emb_cache[key] = get_bi(name).encode(
            [lab_pre + l for l in LABELS], batch_size=128,
            normalize_embeddings=True, show_progress_bar=progress)
    return _emb_cache[key]

1064 en labels from 799 concepts


In [23]:
def chunk_text(text, lo=200, hi=900):
    """Sentence-split, then merge back into blocks of roughly lo..hi characters."""
    text = re.sub(r"\s+", " ", text).strip()
    chunks, buf = [], ""
    for sent in re.split(r"(?<=[.!?])\s+(?=[A-ZÄÖÜ])", text):
        if buf and len(buf) + len(sent) + 1 > hi:
            chunks.append(buf)
            buf = sent
        else:
            buf = f"{buf} {sent}".strip()
        if len(buf) >= lo:
            chunks.append(buf)
            buf = ""
    if buf:
        if chunks and len(buf) < lo:
            chunks[-1] += " " + buf
        else:
            chunks.append(buf)
    return chunks


def retrieve(chunks, top_k=20, bi_model=None):
    """Stage 1 — bi-encoder shortlist: top_k labels per chunk."""
    bi_model = bi_model or BI_MODEL
    _, txt_pre = PREFIXES.get(bi_model, ("", ""))
    emb = get_label_emb(bi_model)
    cos = get_bi(bi_model).encode([txt_pre + c for c in chunks],
                                  normalize_embeddings=True) @ emb.T
    k = min(top_k, len(LABELS) - 1)
    pairs = [(ci, int(li)) for ci in range(len(chunks))
             for li in np.argpartition(-cos[ci], k)[:k]]
    return pairs, cos


def rerank(chunks, pairs, cos, ce_model=None, ce_max_length=256,
           batch_size=32, progress=False):
    """Stage 2 — cross-encoder scores each (label, chunk) pair, best kept per concept."""
    ce = get_ce(ce_model, ce_max_length)
    scores = ce.predict([(LABELS[li], chunks[ci]) for ci, li in pairs],
                        batch_size=batch_size, show_progress_bar=progress)
    best = {}
    for (ci, li), s in zip(pairs, scores):
        cid = CONCEPT_IDS[li]
        if float(s) > best.get(cid, {}).get("ce", -1):
            best[cid] = {"concept": cid, "label": LABELS[li], "ce": float(s),
                         "cos": float(cos[ci, li]), "chunk": ci}
    return sorted(best.values(), key=lambda r: -r["ce"])


def rank_concepts(doc, top_k=20, lo=200, hi=900,
                  bi_model=None, ce_model=None, ce_max_length=256, verbose=True):
    """Full pipeline for one record -> ranked list of dicts."""
    t0 = time.time()
    chunks = chunk_text(doc, lo, hi)
    pairs, cos = retrieve(chunks, top_k, bi_model)
    ranked = rerank(chunks, pairs, cos, ce_model, ce_max_length, progress=verbose)
    if verbose:
        n_lab = len({li for _, li in pairs})
        print(f"{bi_model or BI_MODEL.split('/')[-1]} -> {ce_model or CE_MODEL.split('/')[-1]} | "
              f"{len(doc)} chars, {len(chunks)} chunks, {len(pairs)} CE pairs, "
              f"{n_lab} labels = {n_lab / len(LABELS):.0%} of vocab | {time.time() - t0:.1f}s")
    return ranked


def show(ranked, n=15):
    print(f"  {'ce':>7} {'cos':>6} {'chk':>3}  label")
    for r in ranked[:n]:
        print(f"  {r['ce']:7.4f} {r['cos']:6.3f} {r['chunk']:>3}  "
              f"{r['label']:<34} {r['concept']}")


def compare_models(doc, bi_models, ce_models=(None,), top_k=20, n=10, **kw):
    """Same document through several model pairs; returns {(bi, ce): ranked}."""
    out = {}
    for b in bi_models:
        for c in ce_models:
            print(f"\n===== bi={b or BI_MODEL} | ce={c or CE_MODEL}")
            out[(b, c)] = r = rank_concepts(doc, top_k=top_k, bi_model=b,
                                            ce_model=c, verbose=True, **kw)
            show(r, n)
    return out

In [ ]:
show(rank_concepts(DOC))                                   # defaults
show(rank_concepts(DOC, bi_model="intfloat/multilingual-e5-small"))

results = compare_models(DOC, bi_models=[
    None,                                                     # current MiniLM
    "sentence-transformers/paraphrase-multilingual-mpnet-base-v2",
    "sentence-transformers/LaBSE",
    "intfloat/multilingual-e5-small",
])

encoding 1064 labels with sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2 …
loading bi-encoder sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2 …


Batches: 100%|██████████| 9/9 [00:02<00:00,  3.50it/s]


loading cross-encoder cross-encoder/mmarco-mMiniLMv2-L12-H384-v1 (max_length=256) …


Batches: 100%|██████████| 5/5 [00:02<00:00,  1.70it/s]


paraphrase-multilingual-MiniLM-L12-v2 -> mmarco-mMiniLMv2-L12-H384-v1 | 2196 chars, 8 chunks, 160 CE pairs, 92 labels = 9% of vocab | 13.0s
       ce    cos chk  label
   0.9939  0.749   3  soil organic carbon                SoilOrganicCarbon
   0.9769  0.616   3  soil organic matter content        SoilOrganicMatterContents
   0.9555  0.731   6  soil total carbon                  SoilTotalCarbon
   0.8216  0.693   4  soil carbon density                SoilCarbonDensity
   0.8028  0.627   3  soil organic component             SoilOrganicComponents
   0.7475  0.616   2  soil organic matter                SoilOrganicMatter
   0.6906  0.703   3  soil organic carbon loss           SOCLoss
   0.5926  0.720   3  soil inorganic carbon              SoilInorganicCarbon
   0.5749  0.607   3  critical soil organic matter content CriticalSoilOrganicMatterContents
   0.5098  0.483   1  fertiliser use                     FertiliserUse
   0.4877  0.606   2  soil organic matter class          SoilOrgan

...

sentence-transformers/paraphrase-multilingual-mpnet-base-v2 -> mmarco-mMiniLMv2-L12-H384-v1 | 2196 chars, 8 chunks, 160 CE pairs, 84 labels = 8% of vocab | 22.1s
       ce    cos chk  label
   0.9939  0.714   3  soil organic carbon                SoilOrganicCarbon
   0.9769  0.613   3  soil organic matter content        SoilOrganicMatterContents
   0.9555  0.717   6  soil total carbon                  SoilTotalCarbon
   0.8869  0.595   3  soil contents                      SoilContents
   0.8216  0.689   4  soil carbon density                SoilCarbonDensity
   0.8028  0.585   3  soil organic component             SoilOrganicComponents
   0.7387  0.592   3  soil organic matter                SoilOrganicMatter
   0.6906  0.669   3  soil organic carbon loss           SOCLoss
   0.6397  0.552   4  soil air contents                  SoilAirContents
   0.5926  0.700   3  soil inorganic carbon              SoilInorganicCarbon

===== bi=sentence-transformers/LaBSE | ce=cross-encoder/mmarco-m

Batches: 100%|██████████| 5/5 [00:05<00:00,  1.13s/it]


sentence-transformers/LaBSE -> mmarco-mMiniLMv2-L12-H384-v1 | 2196 chars, 8 chunks, 160 CE pairs, 102 labels = 10% of vocab | 19.8s
       ce    cos chk  label
   0.9939  0.452   3  soil organic carbon                SoilOrganicCarbon
   0.9769  0.413   3  soil organic matter content        SoilOrganicMatterContents
   0.9555  0.399   6  soil total carbon                  SoilTotalCarbon
   0.8028  0.388   3  soil organic component             SoilOrganicComponents
   0.7387  0.384   3  soil organic matter                SoilOrganicMatter
   0.6906  0.442   3  soil organic carbon loss           SOCLoss
   0.6397  0.356   4  soil air contents                  SoilAirContents
   0.6373  0.327   4  soil water contents                SoilWaterContents
   0.6214  0.319   4  soil water content                 SoilMoisture
   0.5749  0.405   3  critical soil organic matter content CriticalSoilOrganicMatterContents

===== bi=intfloat/multilingual-e5-small | ce=cross-encoder/mmarco-mMiniLMv2-L1

Batches: 100%|██████████| 5/5 [00:05<00:00,  1.06s/it]

intfloat/multilingual-e5-small -> mmarco-mMiniLMv2-L12-H384-v1 | 2196 chars, 8 chunks, 160 CE pairs, 75 labels = 7% of vocab | 5.7s
       ce    cos chk  label
   0.9704  0.866   3  soil organic matter contents       SoilOrganicMatterContents
   0.9080  0.887   4  soil contents                      SoilContents
   0.8216  0.877   4  soil carbon density                SoilCarbonDensity
   0.8028  0.867   3  soil organic component             SoilOrganicComponents
   0.6373  0.879   4  soil water contents                SoilWaterContents
   0.6214  0.881   4  soil water content                 SoilMoisture
   0.5098  0.830   1  fertiliser use                     FertiliserUse
   0.4768  0.875   4  soil strength                      SoilStrength
   0.4610  0.878   0  soil organic matter                SoilOrganicMatter
   0.4285  0.876   4  soil clay contents                 SoilClayContents


In [ ]:

results = compare_models(DOC_DE, bi_models=[
    None,
    "intfloat/multilingual-e5-small",                                                    
    "sentence-transformers/paraphrase-multilingual-mpnet-base-v2",
    "sentence-transformers/LaBSE",
    "nomic-ai/nomic-embed-text-v2-moe"
])


===== bi=sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2 | ce=cross-encoder/mmarco-mMiniLMv2-L12-H384-v1


Batches: 100%|██████████| 2/2 [00:00<00:00,  2.18it/s]


paraphrase-multilingual-MiniLM-L12-v2 -> mmarco-mMiniLMv2-L12-H384-v1 | 532 chars, 2 chunks, 40 CE pairs, 40 labels = 4% of vocab | 1.1s
       ce    cos chk  label
   0.5546  0.420   1  land use class                     LandUseClass
   0.3380  0.630   0  soil water contents                SoilWaterContents
   0.2571  0.623   0  soil water flow                    SoilWaterFlow
   0.2232  0.632   0  soil water content                 SoilMoisture
   0.1505  0.616   0  soil water condensation            SoilWaterCondensation
   0.1465  0.419   1  SOC loss                           SOCLoss
   0.1211  0.625   0  water transfer (in soil)           SoilWaterMovement
   0.1085  0.616   0  soil water infiltration            SoilWaterInfiltration
   0.0831  0.628   0  soil water diffusivity             SoilWaterDiffusivity
   0.0636  0.438   1  soil P loss                        SoilPLoss

===== bi=intfloat/multilingual-e5-small | ce=cross-encoder/mmarco-mMiniLMv2-L12-H384-v1


Batches: 100%|██████████| 2/2 [00:00<00:00,  2.67it/s]


intfloat/multilingual-e5-small -> mmarco-mMiniLMv2-L12-H384-v1 | 532 chars, 2 chunks, 40 CE pairs, 40 labels = 4% of vocab | 0.9s
       ce    cos chk  label
   0.5546  0.821   1  land use class                     LandUseClass
   0.2571  0.846   0  soil water flow                    SoilWaterFlow
   0.2205  0.839   0  soil fluid movement                SoilFluidMovement
   0.1498  0.842   0  soil water movement                SoilWaterMovement
   0.1313  0.840   0  soil flow                          SoilFlow
   0.1045  0.841   0  soil fluid transmission            SoilFluidTransmission
   0.0831  0.848   0  soil water diffusivity             SoilWaterDiffusivity
   0.0758  0.843   0  soil moisture redistribution       SoilMoistureRedistribution
   0.0728  0.844   0  soil water adsorption              SoilWaterAdsorption
   0.0719  0.840   0  soil air flow                      SoilAirFlow

===== bi=sentence-transformers/paraphrase-multilingual-mpnet-base-v2 | ce=cross-encoder/mmarco-mM

...

sentence-transformers/paraphrase-multilingual-mpnet-base-v2 -> mmarco-mMiniLMv2-L12-H384-v1 | 532 chars, 2 chunks, 40 CE pairs, 39 labels = 4% of vocab | 1.2s
       ce    cos chk  label
   0.5546  0.547   1  land use class                     LandUseClass
   0.3380  0.578   0  soil water contents                SoilWaterContents
   0.2571  0.609   0  soil water flow                    SoilWaterFlow
   0.2232  0.581   0  soil water content                 SoilMoisture
   0.1702  0.577   0  soil water loss                    SoilWaterLoss
   0.1498  0.588   0  soil water movement                SoilWaterMovement
   0.1313  0.569   0  soil flow                          SoilFlow
   0.1085  0.597   0  soil water infiltration            SoilWaterInfiltration
   0.1003  0.575   1  land use grass                     LandUseGrasses
   0.0920  0.555   1  land use shrubs                    LandUseShrubs

===== bi=sentence-transformers/LaBSE | ce=cross-encoder/mmarco-mMiniLMv2-L12-H384-v1


Batches: 100%|██████████| 2/2 [00:00<00:00,  2.29it/s]


sentence-transformers/LaBSE -> mmarco-mMiniLMv2-L12-H384-v1 | 532 chars, 2 chunks, 40 CE pairs, 38 labels = 4% of vocab | 1.1s
       ce    cos chk  label
   0.5546  0.367   1  land use class                     LandUseClass
   0.2571  0.426   0  soil water flow                    SoilWaterFlow
   0.2232  0.390   0  soil water content                 SoilWaterContents
   0.2232  0.390   0  soil water content                 SoilMoisture
   0.1211  0.497   0  water transfer (in soil)           SoilWaterMovement
   0.1085  0.385   0  soil water infiltration            SoilWaterInfiltration
   0.1003  0.398   1  land use grass                     LandUseGrasses
   0.0920  0.340   1  land use shrubs                    LandUseShrubs
   0.0838  0.392   1  land use forest                    LandUseForests
   0.0831  0.398   0  soil water diffusivity             SoilWaterDiffusivity

===== bi=nomic-ai/nomic-embed-text-v2-moe | ce=cross-encoder/mmarco-mMiniLMv2-L12-H384-v1
encoding 1064 labels 

Loading weights: 100%|██████████| 82/82 [00:00<00:00, 2244.11it/s]
[transformers] NomicBertModel LOAD REPORT from: nomic-ai/nomic-embed-text-v2-moe
Key                                                | Status     | 
---------------------------------------------------+------------+-
layers.{1, 3, 5, 7, 9, 11}.mlp.experts.bias        | UNEXPECTED | 
layers.{0, 2, 4, 6, 8, 10}.mlp.fc1.bias            | UNEXPECTED | 
layers.{0...11}.self_attn.k_proj.bias              | UNEXPECTED | 
layers.{1, 3, 5, 7, 9, 11}.mlp.experts.mlp.w1      | UNEXPECTED | 
layers.{0...11}.self_attn.o_proj.bias              | UNEXPECTED | 
layers.{1, 3, 5, 7, 9, 11}.mlp.router.layer.weight | UNEXPECTED | 
layers.{0...11}.self_attn.v_proj.bias              | UNEXPECTED | 
layers.{0...11}.self_attn.q_proj.bias              | UNEXPECTED | 
layers.{1, 3, 5, 7, 9, 11}.mlp.experts.mlp.w2      | UNEXPECTED | 
layers.{0, 2, 4, 6, 8, 10}.mlp.fc1.weight          | UNEXPECTED | 
layers.{0, 2, 4, 6, 8, 10}.mlp.down_proj.bias   

nomic-ai/nomic-embed-text-v2-moe -> mmarco-mMiniLMv2-L12-H384-v1 | 532 chars, 2 chunks, 40 CE pairs, 30 labels = 3% of vocab | 20.8s
       ce    cos chk  label
   0.9395  0.641   0  phosphorus total elements          PhosphorusTotalElements
   0.4963  0.606   0  soil phosphorus loss               SoilPLoss
   0.2940  0.652   0  phosphorus extractable elements    PhosphorusExtractableElements
   0.1297  0.644   0  phosphorus retention               PhosphorusRetention
   0.0839  0.619   0  gaseous composition of soil air    GaseousCompositionOfSoilAir
   0.0782  0.636   0  flux controlled water infiltration FluxControlledWaterInfiltration
   0.0490  0.613   0  potassium extractable elements     PotassiumExtractableElements
   0.0454  0.619   0  plant water uptake                 PlantWaterUptake
   0.0451  0.658   0  profile controlled water infiltration ProfileControlledWaterInfiltration
   0.0301  0.606   0  rock outcrops cover                RockOutcropCover


nomic-ai/nomic-embed-text-v2-moe has a better performance here.

In [24]:
show(rank_concepts(DOC_DE, bi_model="nomic-ai/nomic-embed-text-v2-moe"), n=10)

encoding 1064 labels with nomic-ai/nomic-embed-text-v2-moe …
loading bi-encoder nomic-ai/nomic-embed-text-v2-moe …


Loading weights: 100%|██████████| 82/82 [00:00<00:00, 5069.47it/s]
[transformers] NomicBertModel LOAD REPORT from: nomic-ai/nomic-embed-text-v2-moe
Key                                                | Status     | 
---------------------------------------------------+------------+-
layers.{1, 3, 5, 7, 9, 11}.mlp.experts.bias        | UNEXPECTED | 
layers.{0, 2, 4, 6, 8, 10}.mlp.fc1.bias            | UNEXPECTED | 
layers.{0...11}.self_attn.k_proj.bias              | UNEXPECTED | 
layers.{1, 3, 5, 7, 9, 11}.mlp.experts.mlp.w1      | UNEXPECTED | 
layers.{0...11}.self_attn.o_proj.bias              | UNEXPECTED | 
layers.{1, 3, 5, 7, 9, 11}.mlp.router.layer.weight | UNEXPECTED | 
layers.{0...11}.self_attn.v_proj.bias              | UNEXPECTED | 
layers.{0...11}.self_attn.q_proj.bias              | UNEXPECTED | 
layers.{1, 3, 5, 7, 9, 11}.mlp.experts.mlp.w2      | UNEXPECTED | 
layers.{0, 2, 4, 6, 8, 10}.mlp.fc1.weight          | UNEXPECTED | 
layers.{0, 2, 4, 6, 8, 10}.mlp.down_proj.bias   

loading cross-encoder cross-encoder/mmarco-mMiniLMv2-L12-H384-v1 (max_length=256) …


Batches: 100%|██████████| 2/2 [00:00<00:00,  2.31it/s]

nomic-ai/nomic-embed-text-v2-moe -> mmarco-mMiniLMv2-L12-H384-v1 | 532 chars, 2 chunks, 40 CE pairs, 32 labels = 3% of vocab | 18.5s
       ce    cos chk  label
   0.0858  0.732   0  soil depositional crusts           SoilDepositionalCrusts
   0.0839  0.744   0  gaseous composition of soil air    GaseousCompositionOfSoilAir
   0.0794  0.750   0  particle density (soil)            SoilDensity
   0.0748  0.742   0  soil gaseous emission              SoilGaseousEmission
   0.0536  0.739   0  soil water evapouration            SoilWaterEvapouration
   0.0174  0.728   0  soil water infiltration capacity   SoilWaterInfiltrationCapacity
   0.0172  0.728   0  soil water infiltration rate numeric SoilWaterInfiltrationRateNumeric
   0.0170  0.666   1  index of crusting                  IndexOfCrusting
   0.0161  0.742   0  soil water holding capacity        SoilWaterHoldingCapacity
   0.0159  0.731   0  soil ecological degradation        SoilEcologicalDegradation


In [ ]:
DOC = """Relative Contribution Of Trees And Crops To Soil Carbon Content In A Parkland System In Burkina Faso Using Variations In Natural C-13 Abundance","The Origin Of Organic Matter Was Studied In The Soils Of A Parkland Of Karite (Vitallaria Paradoxa C.F. Gaertn) And Nere (Parkia Biglobosa (Jacq.) Benth.), Which Is Extensively Cultivated Without The Use Of Fertilisers. In Such Systems, Fertility (Physical, Chemical And Biological) Gradients Around Trees Have Been Attributed By Some Authors To A Priori Differences In Fertility, Allowing For Better Tree Establishment On Richer Sites. In Reverse, Other Workers Believed That These Gradients Are Due To The Contribution Of Trees To The Formation Of Soil Organic Matter Through Litter And Decay Of Roots. Measurements Of The Variations In The C-13 Isotopic Composition Allowed For A Distinction Between Tree (C-3) Derived C And Crop And Grass (C-4) Derived C In The Total Soil Organic C Content. The Organic Carbon Contents Of The Soils Were Recorded Under The Two Species At Two Soil Depths And At Five Distances Going From Tree Trunk To The Open Area And Their C Isotopic Signatures Were Analysed. The Results Showed That Soil Carbon Contents Under Karite (6.43 +/- 0.45 G Kg(-1)) And Nere (5.65 +/- 0.27 G Kg(-1)) Were Significantly Higher (P < 0.01) Than In The Open Area (4.09 +/- 0.26 G Kg(-1)). The Delta C-13 Of Soil C Was Significantly Higher (P < 0.001) In The Open Area (-17.5 +/- 0.3 Parts Per Thousand) Compared With The Values Obtained On Average With Depth And Distance From Tree Under Karite (-20.2 +/- 0.4 Parts Per Thousand) And Nere (-20.1 +/- 0.4 Parts Per Thousand). The C-4-Derived Soil C Was Approximately Constant, And The Differences In Total Soil C Were Fully Explained By The C-3 (Tree) Contributions To Soil Carbon Of 4.01 +/- 0.71, 3.02 +/- 0.53, 1.53 +/- 0.10 G Kg(-1), Respectively Under Karite, Nere And In The Open Area. These Results Show That Trees In Parklands Have A Directly Positive Contribution To Soil Carbon Content, Justifying The Need To Encourage The Maintenance Of Trees In These Systems In Semi-Arid Environments Where The Carbon Content Of Soil Appears To Be The First Limiting Factor For Crop Growth.
"""
show(rank_concepts(DOC, bi_model="nomic-ai/nomic-embed-text-v2-moe"), n=10)

Batches: 100%|██████████| 5/5 [00:03<00:00,  1.55it/s]

nomic-ai/nomic-embed-text-v2-moe -> mmarco-mMiniLMv2-L12-H384-v1 | 2196 chars, 8 chunks, 160 CE pairs, 67 labels = 6% of vocab | 4.1s
       ce    cos chk  label
   0.5523  0.530   3  critical soil organic matter contents CriticalSoilOrganicMatterContents
   0.1428  0.556   4  soil particle size distribution    SoilParticleSizeDistribution
   0.1246  0.497   4  sodium total elements              SodiumTotalElements
   0.0941  0.529   3  gaseous composition of soil air    GaseousCompositionOfSoilAir
   0.0938  0.567   6  soil aggregate size distribution   SoilAggregateSizeDistribution
   0.0922  0.499   4  soil particle specific surface area SoilParticleSpecificSurfaceArea
   0.0877  0.503   4  soil water evapouration rate       SoilWaterEvapourationRate
   0.0831  0.505   4  soil specific gravity              SoilSpecificGravity
   0.0821  0.501   4  soil aggregation degree            SoilAggregationDegree
   0.0713  0.503   4  aggregate density (soil)           SoilDensity


But nomic-ai/nomic-embed-text-v2-moe has a bad performance for the english text. Now try to use the paraphrase-multilingual-MiniLM-L12-v2 and chunk the text differently.

In [33]:
show(rank_concepts(DOC_DE, bi_model="paraphrase-multilingual-MiniLM-L12-v2", lo = 100), n=10)

Batches: 100%|██████████| 2/2 [00:00<00:00,  2.48it/s]

paraphrase-multilingual-MiniLM-L12-v2 -> mmarco-mMiniLMv2-L12-H384-v1 | 532 chars, 2 chunks, 40 CE pairs, 40 labels = 4% of vocab | 1.0s
       ce    cos chk  label
   0.5546  0.420   1  land use class                     LandUseClass
   0.3380  0.630   0  soil water contents                SoilWaterContents
   0.2571  0.623   0  soil water flow                    SoilWaterFlow
   0.2232  0.632   0  soil water content                 SoilMoisture
   0.1505  0.616   0  soil water condensation            SoilWaterCondensation
   0.1465  0.419   1  SOC loss                           SOCLoss
   0.1211  0.625   0  water transfer (in soil)           SoilWaterMovement
   0.1085  0.616   0  soil water infiltration            SoilWaterInfiltration
   0.0831  0.628   0  soil water diffusivity             SoilWaterDiffusivity
   0.0636  0.438   1  soil P loss                        SoilPLoss


If increase lo, result has less diversity.

In [36]:
DOC_1 = """ 
Our study reveals the effects of GCFs on a soil-crop system: in general, with an increasing number of GCFs, soil properties, and plant biomass reacted negatively. For example, the higher the level of GCFs, the lower the plant biomass and soil water stable aggregation. We also find that MP applied as a single factor had minimal effects on soil properties and crop growth. However, when combined with other individual factors, it significantly altered the effect size, sometimes even causing directional change. Our results revealed that the interaction between MP and other GCFs is not an additive response. Due to the characteristics of MP, the interaction mechanism between heavy metal and MP is obviously different from the response between drought and MP, and their combined effects on the soil-plant system may fundamentally vary Factor interactions are key to understanding and predicting how GCFs influence soil and plants. With an increasing number of GCFs involved, it becomes more and more complicated to predict effects on ecosystems. Our study is among the first to systematically examine how microplastic acts in combination with a range of other important environmental drivers, and thus offers a first step toward understanding these elusive interactive effects.
"""
show(rank_concepts(DOC_1, bi_model="paraphrase-multilingual-MiniLM-L12-v2"), n=10)

Batches: 100%|██████████| 3/3 [00:01<00:00,  2.25it/s]

paraphrase-multilingual-MiniLM-L12-v2 -> mmarco-mMiniLMv2-L12-H384-v1 | 1281 chars, 4 chunks, 80 CE pairs, 65 labels = 6% of vocab | 1.6s
       ce    cos chk  label
   0.0345  0.512   1  soil function indicators           SoilFunctionIndicators
   0.0292  0.582   1  soil volume change                 SoilVolumeChange
   0.0272  0.560   1  soil structural change             SoilStructuralChange
   0.0252  0.523   1  soil reactivity                    SoilReactivity
   0.0226  0.505   1  soil moisture deficit              SoilMoistureDeficit
   0.0224  0.549   1  soil shrinkage                     SoilShrinkage
   0.0152  0.537   1  soil shrinking                     SoilShrinking
   0.0115  0.510   1  soil structural decline            SoilStructuralDecline
   0.0115  0.547   1  soil swell shrink property         SoilSwellShrinkProperties
   0.0101  0.503   1  soil permittivity                  SoilPermittivity
